In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR        = r"C:\Users\AD\Downloads\datathon-2026-round-1\dataset"
SALES_FILE      = os.path.join(DATA_DIR, "sales.csv")
SUBMISSION_FILE = os.path.join(DATA_DIR, "sample_submission.csv")
WEB_TRAFFIC     = os.path.join(DATA_DIR, "web_traffic.csv")
PROMOS_FILE     = os.path.join(DATA_DIR, "promotions.csv")
OUTPUT_FILE     = os.path.join(os.getcwd(), "submission.csv")
FEATIMP_FILE    = os.path.join(os.getcwd(), "feature_importance_v6.csv")

OPTUNA_TRIALS = 20
N_SEEDS = 3

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ─────────────────── 1. LOAD + QA ───────────────────
print("1. LOAD + DATA QUALITY")
print("=" * 70)
train = pd.read_csv(SALES_FILE, parse_dates=["Date"])
test  = pd.read_csv(SUBMISSION_FILE, parse_dates=["Date"])
print(f"  Train: {len(train):>5}  {train['Date'].min().date()} -> {train['Date'].max().date()}")
print(f"  Test : {len(test):>5}  {test['Date'].min().date()} -> {test['Date'].max().date()}")

full_range = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")
if len(full_range.difference(train["Date"])) > 0:
    train = (train.set_index("Date").reindex(full_range)
                  .rename_axis("Date").reset_index())
for c in ["Revenue", "COGS"]:
    train[c] = train[c].interpolate("linear").bfill().ffill()

# Business-rule fix: COGS must be < Revenue (per products.csv ràng buộc)
flip = (train["COGS"] > train["Revenue"]).sum()
if flip > 0:
    print(f"  Clipped {flip} rows with COGS > Revenue to 0.98 * Revenue")
    m = train["COGS"] > train["Revenue"]
    train.loc[m, "COGS"] = train.loc[m, "Revenue"] * 0.98

# Winsorize at 99.9%
for c in ["Revenue", "COGS"]:
    hi = train[c].quantile(0.999)
    n = (train[c] > hi).sum()
    train[c] = train[c].clip(lower=0.0, upper=hi)
    if n:
        print(f"  Winsorized {c}: {n} values capped at {hi:,.0f}")

1. LOAD + DATA QUALITY
  Train:  3833  2012-07-04 -> 2022-12-31
  Test :   548  2023-01-01 -> 2024-07-01
  Clipped 382 rows with COGS > Revenue to 0.98 * Revenue
  Winsorized Revenue: 4 values capped at 17,547,616
  Winsorized COGS: 4 values capped at 15,222,869


In [3]:
# ─────────────────── 2. EXTERNAL MERGES ───────────────────
wt = pd.read_csv(WEB_TRAFFIC, parse_dates=["date"])
wt_daily = (wt.groupby("date")
              .agg(sessions=("sessions", "sum"),
                   page_views=("page_views", "sum"))
              .reset_index().rename(columns={"date": "Date"}))

promos = pd.read_csv(PROMOS_FILE, parse_dates=["start_date", "end_date"])
promos["discount_value"] = promos["discount_value"].fillna(0)

all_dates = pd.concat([train[["Date"]], test[["Date"]]]).drop_duplicates().sort_values("Date")
promo_rows = []
for d in all_dates["Date"]:
    a = promos[(promos["start_date"] <= d) & (promos["end_date"] >= d)]
    promo_rows.append({
        "Date": d,
        "n_active_promos": len(a),
        "max_discount": a["discount_value"].max() if len(a) else 0,
    })
promo_df = pd.DataFrame(promo_rows)

In [ ]:
# ─────────────────── 3. FEATURE ENGINEERING ───────────────────
print("\n2. FEATURE ENGINEERING")
print("=" * 70)

# v5 had lag_1 → 24% importance → cascades in recursion. Drop it.
LAGS  = [7, 14, 28, 90, 365, 730]
ROLLS = [7, 14, 28]
EWMA_SPANS = [7, 14, 28]


def _is_tet(d):
    return (d.month == 1 and d.day >= 20) or (d.month == 2 and d.day <= 15)


def _is_vn_holiday(d):
    md = (d.month, d.day)
    if md in [(1, 1), (4, 30), (5, 1), (9, 2), (12, 25)]:
        return True
    return _is_tet(d)


def _is_flash(d):
    md = (d.month, d.day)
    return ((d.month == 11 and 10 <= d.day <= 12) or
            (d.month == 12 and 11 <= d.day <= 13) or
            (d.month == 6  and 17 <= d.day <= 18) or
            (d.month == 8  and 7  <= d.day <= 9))


def _proximity(dates, is_event_fn, cap=60):
    """Days to next / from last event on an ordered date series, clamped to cap."""
    n = len(dates)
    ev = np.array([is_event_fn(d) for d in dates], dtype=bool)
    to_next = np.full(n, cap, dtype=np.int32)
    last = cap
    for i in range(n - 1, -1, -1):
        last = 0 if ev[i] else min(last + 1, cap)
        to_next[i] = last
    from_last = np.full(n, cap, dtype=np.int32)
    last = cap
    for i in range(n):
        last = 0 if ev[i] else min(last + 1, cap)
        from_last[i] = last
    return to_next, from_last


def add_calendar(df):
    df = df.copy()
    df["year"]    = df["Date"].dt.year
    df["month"]   = df["Date"].dt.month
    df["day"]     = df["Date"].dt.day
    df["dow"]     = df["Date"].dt.dayofweek
    df["doy"]     = df["Date"].dt.dayofyear
    df["week"]    = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["Date"].dt.quarter
    df["is_weekend"]     = (df["dow"] >= 5).astype(int)
    df["is_working_day"] = (df["dow"] < 5).astype(int)

    df["sin_doy"] = np.sin(2*np.pi*df["doy"]/365.25)
    df["cos_doy"] = np.cos(2*np.pi*df["doy"]/365.25)
    df["sin_dow"] = np.sin(2*np.pi*df["dow"]/7)
    df["cos_dow"] = np.cos(2*np.pi*df["dow"]/7)
    df["sin_mo"]  = np.sin(2*np.pi*df["month"]/12)
    df["cos_mo"]  = np.cos(2*np.pi*df["month"]/12)

    dim = df["Date"].dt.days_in_month
    df["days_to_month_end"]  = (dim - df["day"]).astype(int)
    df["is_month_end_3d"]    = (df["days_to_month_end"] <= 2).astype(int)
    df["is_month_start_3d"]  = (df["day"] <= 3).astype(int)

    df["is_tet"]     = df["Date"].apply(_is_tet).astype(int)
    df["is_holiday"] = df["Date"].apply(_is_vn_holiday).astype(int)
    df["is_flash"]   = df["Date"].apply(_is_flash).astype(int)

    # NEW: proximity features — non-cascading (test knows its own calendar)
    dates = df["Date"].tolist()
    for name, fn in (("holiday", _is_vn_holiday), ("tet", _is_tet), ("flash", _is_flash)):
        to_next, from_last = _proximity(dates, fn)
        df[f"days_to_next_{name}"]  = to_next
        df[f"days_from_last_{name}"] = from_last
    return df


def add_lag_roll(df, col, pre):
    df = df.copy()
    for l in LAGS:
        df[f"{pre}lag_{l}"] = df[col].shift(l)
    shifted = df[col].shift(1)
    for w in ROLLS:
        r = shifted.rolling(w, min_periods=1)
        df[f"{pre}rmean_{w}"] = r.mean()
        df[f"{pre}rstd_{w}"]  = r.std()
        df[f"{pre}rmin_{w}"]  = r.min()
        df[f"{pre}rmax_{w}"]  = r.max()
    for s in EWMA_SPANS:
        df[f"{pre}ewm_{s}"] = shifted.ewm(span=s, min_periods=1).mean()
    df[f"{pre}yoy_avg"] = (df[col].shift(365).fillna(0) + df[col].shift(730).fillna(0)) / 2
    return df


df = (train.merge(wt_daily, on="Date", how="left")
           .merge(promo_df, on="Date", how="left"))
for c in ["sessions", "page_views"]:
    df[c] = df[c].ffill().bfill().fillna(df[c].median())
df["n_active_promos"] = df["n_active_promos"].fillna(0)
df["max_discount"]    = df["max_discount"].fillna(0)

df = add_calendar(df)
df = add_lag_roll(df, "Revenue", "rev_")
df = add_lag_roll(df, "COGS",    "cogs_")

for c in ["sessions", "page_views"]:
    df[f"{c}_r7"]  = df[c].shift(1).rolling(7,  min_periods=1).mean()
    df[f"{c}_r28"] = df[c].shift(1).rolling(28, min_periods=1).mean()


In [ ]:
# ─────────────────── 4. GROUP-MEAN ENCODINGS ───────────────────
# Mean log-Revenue by calendar groups, computed from TRAINING PORTION only
# (we'll recompute per-fold in CV, and on 2013→2021 for the 2022 holdout eval).
GROUP_KEYS = [
    ("dow",),
    ("month",),
    ("quarter",),
    ("dow", "month"),
    ("dow", "quarter"),
    ("week",),
]


def add_group_means(base_df, source_df, keys_list):
    """Attach mean log-Revenue group encodings from source_df → base_df."""
    base = base_df.copy()
    src = source_df[["Revenue"] + [c for c in set(sum((list(k) for k in keys_list), []))
                                   if c in source_df.columns]].copy()
    src["_logrev"] = np.log1p(src["Revenue"].clip(lower=0))
    for keys in keys_list:
        gmean = src.groupby(list(keys))["_logrev"].mean().rename(
            f"gmean_logrev_{'_'.join(keys)}")
        base = base.merge(gmean, on=list(keys), how="left")
    return base


eng_cols_for_na = [c for c in df.columns
                   if any(k in c for k in ["lag_", "rmean_", "rstd_", "rmin_", "rmax_",
                                           "ewm_", "yoy_", "_r7", "_r28"])]
for c in eng_cols_for_na:
    df[c] = df[c].fillna(df[c].median())
if df.drop(columns=["Date"]).isnull().any().any():
    df = df.fillna(0)

print( df.shape)


In [ ]:
# ─────────────────── 5. SPLIT ───────────────────
exclude_base = ["Date", "Revenue", "COGS"]
min_date = train["Date"].min() + pd.Timedelta(days=365)
df_model = df[df["Date"] >= min_date].copy()

val_mask = df_model["Date"].dt.year == 2022
df_tr_raw = df_model[~val_mask].copy()
df_vl_raw = df_model[val_mask].copy()

# Group means from train portion only (no leakage into val)
df_tr = add_group_means(df_tr_raw, df_tr_raw, GROUP_KEYS)
df_vl = add_group_means(df_vl_raw, df_tr_raw, GROUP_KEYS)
gmean_cols = [c for c in df_tr.columns if c.startswith("gmean_logrev_")]
for c in gmean_cols:
    med = df_tr[c].median()
    df_tr[c] = df_tr[c].fillna(med)
    df_vl[c] = df_vl[c].fillna(med)

print(f"  Train: {len(df_tr)}  {df_tr['Date'].min().date()} -> {df_tr['Date'].max().date()}  ({df_tr.shape[1]} cols)")
print(f"  Val  : {len(df_vl)}  {df_vl['Date'].min().date()} -> {df_vl['Date'].max().date()}")

all_feat  = [c for c in df_tr.columns if c not in exclude_base]
rev_feats  = [c for c in all_feat if not c.startswith("cogs_")]
rate_feats = list(all_feat)

y_tr_rev_log = np.log1p(df_tr["Revenue"].values)
y_vl_rev_log = np.log1p(df_vl["Revenue"].values)
y_vl_rev     = df_vl["Revenue"].values
eps = 1.0
rate_tr = df_tr["COGS"].values / (df_tr["Revenue"].values + eps)
rate_vl = df_vl["COGS"].values / (df_vl["Revenue"].values + eps)

X_tr_rev  = df_tr[rev_feats].values
X_vl_rev  = df_vl[rev_feats].values
X_tr_rate = df_tr[rate_feats].values
X_vl_rate = df_vl[rate_feats].values


In [ ]:
# ─────────────────── 6. OPTUNA HP SEARCH ───────────────────
print("\n3. OPTUNA TUNING ")
print("=" * 70)


def ts_cv_score(model_fn, X, y, n_splits=4):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr_i, vl_i in tscv.split(X):
        m = model_fn()
        if hasattr(m, "fit") and "eval_set" in m.fit.__code__.co_varnames:
            # XGB / LGB path
            try:
                m.fit(X[tr_i], y[tr_i], eval_set=[(X[vl_i], y[vl_i])], verbose=False)
            except TypeError:
                m.fit(X[tr_i], y[tr_i])
        else:
            m.fit(X[tr_i], y[tr_i])
        scores.append(np.sqrt(mean_squared_error(y[vl_i], m.predict(X[vl_i]))))
    return float(np.mean(scores))


def opt_lgb(X, y, n_trials=OPTUNA_TRIALS):
    def objective(trial):
        params = dict(
            n_estimators=1200,
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 63),
            max_depth=trial.suggest_int("max_depth", 4, 10),
            min_child_samples=trial.suggest_int("min_child_samples", 15, 60),
            subsample=trial.suggest_float("subsample", 0.6, 0.95),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 0.95),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
            random_state=SEED, n_jobs=-1, verbose=-1,
        )
        return ts_cv_score(lambda: lgb.LGBMRegressor(**params), X, y, n_splits=4)
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = {**study.best_params,
            "n_estimators": 3000, "random_state": SEED, "n_jobs": -1, "verbose": -1}
    return best, study.best_value


def opt_xgb(X, y, n_trials=OPTUNA_TRIALS):
    def objective(trial):
        params = dict(
            n_estimators=1200,
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            max_depth=trial.suggest_int("max_depth", 4, 10),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
            subsample=trial.suggest_float("subsample", 0.6, 0.95),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 0.95),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
            random_state=SEED, n_jobs=-1, verbosity=0,
        )
        return ts_cv_score(lambda: XGBRegressor(**params), X, y, n_splits=4)
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = {**study.best_params,
            "n_estimators": 3000, "random_state": SEED, "n_jobs": -1, "verbosity": 0,
            "early_stopping_rounds": 100}
    return best, study.best_value


CAT_PARAMS = dict(
    iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=5.0,
    subsample=0.8, rsm=0.7, random_seed=SEED, loss_function="RMSE",
    bootstrap_type="Bernoulli", verbose=False, early_stopping_rounds=100,
)

print("  Revenue: tuning LGB...")
LGB_REV, lgb_rev_cv = opt_lgb(X_tr_rev, y_tr_rev_log)
print(f"    best CV RMSE = {lgb_rev_cv:.4f}")
print("  Revenue: tuning XGB...")
XGB_REV, xgb_rev_cv = opt_xgb(X_tr_rev, y_tr_rev_log)
print(f"    best CV RMSE = {xgb_rev_cv:.4f}")

print("  COGS rate: tuning LGB...")
LGB_RATE, lgb_rate_cv = opt_lgb(X_tr_rate, rate_tr)
print(f"    best CV RMSE = {lgb_rate_cv:.4f}")
print("  COGS rate: tuning XGB...")
XGB_RATE, xgb_rate_cv = opt_xgb(X_tr_rate, rate_tr)
print(f"    best CV RMSE = {xgb_rate_cv:.4f}")

In [ ]:
# ─────────────────── 7. BAKEOFF WITH TUNED PARAMS ───────────────────
print("\n4. BAKEOFF ON 2022 HOLDOUT (tuned)")
print("=" * 70)


def fit_lgb(X_tr, y_tr, X_vl, y_vl, params):
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m


def fit_cat(X_tr, y_tr, X_vl, y_vl, params=CAT_PARAMS, seed=SEED):
    p = {**params, "random_seed": seed}
    m = CatBoostRegressor(**p)
    m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), use_best_model=True)
    return m


def fit_xgb(X_tr, y_tr, X_vl, y_vl, params):
    m = XGBRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    return m


def bakeoff(X_tr, y_tr, X_vl, y_vl, params_lgb, params_xgb, tag):
    print(f"\n  --- {tag} ---")
    m_lgb = fit_lgb(X_tr, y_tr, X_vl, y_vl, params_lgb)
    m_cat = fit_cat(X_tr, y_tr, X_vl, y_vl)
    m_xgb = fit_xgb(X_tr, y_tr, X_vl, y_vl, params_xgb)

    # Ridge baseline (scaled)
    sc = StandardScaler().fit(X_tr)
    m_rid = Ridge(alpha=10.0).fit(sc.transform(X_tr), y_tr)

    preds = {
        "ridge": m_rid.predict(sc.transform(X_vl)),
        "lgb":   m_lgb.predict(X_vl),
        "cat":   m_cat.predict(X_vl),
        "xgb":   m_xgb.predict(X_vl),
    }
    for k, p in preds.items():
        rmse = np.sqrt(mean_squared_error(y_vl, p))
        print(f"    {k:5s}  val RMSE={rmse:.4f}")
    return {"models": {"ridge": (sc, m_rid), "lgb": m_lgb, "cat": m_cat, "xgb": m_xgb},
            "preds": preds}


rev_bakeoff  = bakeoff(X_tr_rev,  y_tr_rev_log, X_vl_rev,  y_vl_rev_log,
                       LGB_REV,  XGB_REV,  "Revenue (log)")
rate_bakeoff = bakeoff(X_tr_rate, rate_tr,      X_vl_rate, rate_vl,
                       LGB_RATE, XGB_RATE, "COGS rate")


In [ ]:
# ─────────────────── 8. OOF-WEIGHTED BLEND ───────────────────
# Use TimeSeriesSplit CV RMSE as the weight 
print("\n5. OOF-WEIGHTED BLEND")
print("=" * 70)


def oof_rmse(name, X, y, params, n_splits=4):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr_i, vl_i in tscv.split(X):
        if name == "lgb":
            m = lgb.LGBMRegressor(**{**params, "n_estimators": 1200})
            m.fit(X[tr_i], y[tr_i], eval_set=[(X[vl_i], y[vl_i])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        elif name == "cat":
            m = CatBoostRegressor(**{**CAT_PARAMS, "iterations": 1200})
            m.fit(X[tr_i], y[tr_i], eval_set=(X[vl_i], y[vl_i]), use_best_model=True)
        elif name == "xgb":
            m = XGBRegressor(**{**params, "n_estimators": 1200})
            m.fit(X[tr_i], y[tr_i], eval_set=[(X[vl_i], y[vl_i])], verbose=False)
        scores.append(np.sqrt(mean_squared_error(y[vl_i], m.predict(X[vl_i]))))
    return float(np.mean(scores))


print("  Revenue OOF RMSEs:")
rev_oof = {
    "lgb": oof_rmse("lgb", X_tr_rev, y_tr_rev_log, LGB_REV),
    "cat": oof_rmse("cat", X_tr_rev, y_tr_rev_log, None),
    "xgb": oof_rmse("xgb", X_tr_rev, y_tr_rev_log, XGB_REV),
}
print(f"    {rev_oof}")
print("  COGS-rate OOF RMSEs:")
rate_oof = {
    "lgb": oof_rmse("lgb", X_tr_rate, rate_tr, LGB_RATE),
    "cat": oof_rmse("cat", X_tr_rate, rate_tr, None),
    "xgb": oof_rmse("xgb", X_tr_rate, rate_tr, XGB_RATE),
}
print(f"    {rate_oof}")


def inv_rmse_weights(oof):
    w = {k: 1 / v for k, v in oof.items()}
    s = sum(w.values())
    return {k: v / s for k, v in w.items()}


rev_w  = inv_rmse_weights(rev_oof)
rate_w = inv_rmse_weights(rate_oof)
print(f"  Revenue blend weights: {rev_w}")
print(f"  COGS-rate blend weights: {rate_w}")

# Validate blend
pred_vl_rev_log = sum(rev_w[k] * rev_bakeoff["preds"][k] for k in ["lgb", "cat", "xgb"])
pred_vl_rev = np.clip(np.expm1(pred_vl_rev_log), 0, None)
pred_vl_rate = np.clip(sum(rate_w[k] * rate_bakeoff["preds"][k] for k in ["lgb", "cat", "xgb"]),
                       0.3, 1.0)
pred_vl_cogs = pred_vl_rev * pred_vl_rate
y_vl_cogs = df_vl["COGS"].values


def report(name, y, p):
    mae  = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    r2   = r2_score(y, p)
    mape = np.mean(np.abs((y - p) / np.where(y == 0, 1, y))) * 100
    print(f"  {name}:  MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R2={r2:.4f}  MAPE={mape:5.2f}%")


print()
report("Revenue", y_vl_rev,  pred_vl_rev)
report("COGS   ", y_vl_cogs, pred_vl_cogs)

In [ ]:
# ─────────────────── 10. REFIT ON FULL + SEED AVG ───────────────────
print("\n7. REFIT ON FULL DATA, SEED-AVERAGED")
print("=" * 70)

df_full = add_group_means(df_model, df_model, GROUP_KEYS)
for c in gmean_cols:
    df_full[c] = df_full[c].fillna(df_full[c].median())
# Re-sync feature order (group cols may re-attach in different order)
rev_feats  = [c for c in df_full.columns if c not in exclude_base and not c.startswith("cogs_")]
rate_feats = [c for c in df_full.columns if c not in exclude_base]

X_full_rev  = df_full[rev_feats].values
X_full_rate = df_full[rate_feats].values
y_full_rev_log = np.log1p(df_full["Revenue"].values)
y_full_rate    = df_full["COGS"].values / (df_full["Revenue"].values + eps)


def best_iter_or(obj, kind, default):
    if kind == "lgb":  return getattr(obj, "best_iteration_", None) or default
    if kind == "cat":  return obj.get_best_iteration() or default
    if kind == "xgb":  return getattr(obj, "best_iteration", None) or default
    return default


def seed_avg_fit(name, X, y, base_params, best_n, seeds):
    """Return list of (seeds) fitted models."""
    models = []
    for s in seeds:
        if name == "lgb":
            p = {**base_params, "n_estimators": best_n, "random_state": s,
                 "bagging_seed": s, "feature_fraction_seed": s}
            m = lgb.LGBMRegressor(**p); m.fit(X, y); models.append(m)
        elif name == "cat":
            p = {**{k: v for k, v in CAT_PARAMS.items()
                    if k not in ("iterations", "early_stopping_rounds")},
                 "iterations": best_n, "random_seed": s}
            m = CatBoostRegressor(**p); m.fit(X, y); models.append(m)
        elif name == "xgb":
            p = {**{k: v for k, v in base_params.items() if k != "early_stopping_rounds"},
                 "n_estimators": best_n, "random_state": s}
            m = XGBRegressor(**p); m.fit(X, y, verbose=False); models.append(m)
    return models


SEEDS = [SEED, SEED + 101, SEED + 202][:N_SEEDS]

best_n_rev = {
    "lgb": best_iter_or(rev_bakeoff["models"]["lgb"], "lgb", 2000),
    "cat": best_iter_or(rev_bakeoff["models"]["cat"], "cat", 2000),
    "xgb": best_iter_or(rev_bakeoff["models"]["xgb"], "xgb", 2000),
}
best_n_rate = {
    "lgb": best_iter_or(rate_bakeoff["models"]["lgb"], "lgb", 2000),
    "cat": best_iter_or(rate_bakeoff["models"]["cat"], "cat", 2000),
    "xgb": best_iter_or(rate_bakeoff["models"]["xgb"], "xgb", 2000),
}

rev_full = {
    "lgb": seed_avg_fit("lgb", X_full_rev, y_full_rev_log, LGB_REV, best_n_rev["lgb"], SEEDS),
    "cat": seed_avg_fit("cat", X_full_rev, y_full_rev_log, None,    best_n_rev["cat"], SEEDS),
    "xgb": seed_avg_fit("xgb", X_full_rev, y_full_rev_log, XGB_REV, best_n_rev["xgb"], SEEDS),
}
rate_full = {
    "lgb": seed_avg_fit("lgb", X_full_rate, y_full_rate, LGB_RATE, best_n_rate["lgb"], SEEDS),
    "cat": seed_avg_fit("cat", X_full_rate, y_full_rate, None,     best_n_rate["cat"], SEEDS),
    "xgb": seed_avg_fit("xgb", X_full_rate, y_full_rate, XGB_RATE, best_n_rate["xgb"], SEEDS),
}
print(f"  Seed-averaged: {len(SEEDS)} seeds per model × 3 models × 2 targets")

In [ ]:
# ─────────────────── 11. RECURSIVE INFERENCE ───────────────────
print("\n8. RECURSIVE INFERENCE")
print("=" * 70)

full_series = pd.concat([
    train[["Date", "Revenue", "COGS"]],
    test[["Date"]].assign(Revenue=np.nan, COGS=np.nan),
], ignore_index=True).sort_values("Date").reset_index(drop=True)
full_series = (full_series.merge(wt_daily, on="Date", how="left")
                          .merge(promo_df, on="Date", how="left"))

ext_cols = ["sessions", "page_views", "n_active_promos", "max_discount"]
hist_mask = full_series["Date"] <= train["Date"].max()
history_idx = full_series.loc[hist_mask].set_index("Date")
month_mean = (full_series.loc[hist_mask]
              .assign(_m=lambda d: d["Date"].dt.month)
              .groupby("_m")[ext_cols].mean())

for i in full_series.index[full_series["Date"] > train["Date"].max()]:
    d = full_series.at[i, "Date"]
    for yrs_back in (1, 2, 3):
        cand = d - pd.DateOffset(years=yrs_back)
        if cand in history_idx.index:
            for c in ext_cols:
                if pd.isna(full_series.at[i, c]):
                    v = history_idx.at[cand, c]
                    if pd.notna(v):
                        full_series.at[i, c] = v
            break
    m = d.month
    for c in ext_cols:
        if pd.isna(full_series.at[i, c]) and m in month_mean.index:
            full_series.at[i, c] = month_mean.at[m, c]
for c in ext_cols:
    full_series[c] = full_series[c].fillna(0)

full_series = add_calendar(full_series)
# Merge in the trained group-mean encodings
gmean_src = df_full[list({*sum((list(k) for k in GROUP_KEYS), []), *gmean_cols})].drop_duplicates()
# Re-compute group means from df_full as canonical source
for keys in GROUP_KEYS:
    col = f"gmean_logrev_{'_'.join(keys)}"
    src = df_full[[*keys, col]].drop_duplicates()
    full_series = full_series.merge(src, on=list(keys), how="left")
for c in gmean_cols:
    med = df_full[c].median()
    full_series[c] = full_series[c].fillna(med)

test_dates  = test["Date"].tolist()
date_arr    = full_series["Date"].values
rev_arr     = full_series["Revenue"].values.copy()
cogs_arr    = full_series["COGS"].values.copy()
date_to_idx = {pd.Timestamp(d): i for i, d in enumerate(date_arr)}
sess_arr    = full_series["sessions"].values
pv_arr      = full_series["page_views"].values


def hist_lag_roll(idx, arr, pre, row):
    for l in LAGS:
        j = idx - l
        row[f"{pre}lag_{l}"] = arr[j] if j >= 0 else np.nan
    past = arr[:idx]
    past = past[~np.isnan(past)]
    for w in ROLLS:
        tail = past[-w:] if len(past) else past
        row[f"{pre}rmean_{w}"] = np.mean(tail) if len(tail) else np.nan
        row[f"{pre}rstd_{w}"]  = np.std(tail)  if len(tail) > 1 else 0.0
        row[f"{pre}rmin_{w}"]  = np.min(tail)  if len(tail) else np.nan
        row[f"{pre}rmax_{w}"]  = np.max(tail)  if len(tail) else np.nan
    if len(past):
        s = pd.Series(past)
        for span in EWMA_SPANS:
            row[f"{pre}ewm_{span}"] = float(s.ewm(span=span, min_periods=1).mean().iloc[-1])
    else:
        for span in EWMA_SPANS:
            row[f"{pre}ewm_{span}"] = 0.0
    yoys = [arr[idx - s] for s in (365, 730) if idx >= s and not np.isnan(arr[idx - s])]
    row[f"{pre}yoy_avg"] = np.mean(yoys) if yoys else 0.0


def row_to_vec(row, cols):
    vals = []
    for c in cols:
        v = row.get(c, 0.0)
        if isinstance(v, pd.Timestamp) or pd.isna(v):
            v = 0.0
        vals.append(v)
    return np.asarray(vals, dtype=np.float64).reshape(1, -1)


def seed_avg_predict(models_list, X):
    preds = [m.predict(X)[0] for m in models_list]
    return float(np.mean(preds))


def blend_predict(models_dict, X, weights, invert_log=False):
    out = 0.0
    for k, lst in models_dict.items():
        out += weights[k] * seed_avg_predict(lst, X)
    return float(np.expm1(out)) if invert_log else out


print(f"  Predicting {len(test_dates)} days...")
for i, tdate in enumerate(test_dates):
    idx = date_to_idx[pd.Timestamp(tdate)]
    row = full_series.loc[idx].to_dict()

    hist_lag_roll(idx, rev_arr,  "rev_",  row)
    hist_lag_roll(idx, cogs_arr, "cogs_", row)

    for col, arr in (("sessions", sess_arr), ("page_views", pv_arr)):
        s7  = max(0, idx - 7)
        s28 = max(0, idx - 28)
        row[f"{col}_r7"]  = np.mean(arr[s7:idx])  if idx > s7  else arr[idx]
        row[f"{col}_r28"] = np.mean(arr[s28:idx]) if idx > s28 else arr[idx]

    x_rev  = row_to_vec(row, rev_feats)
    x_rate = row_to_vec(row, rate_feats)

    p_rev  = max(0.0, blend_predict(rev_full,  x_rev,  rev_w,  invert_log=True))
    p_rate = float(np.clip(blend_predict(rate_full, x_rate, rate_w), 0.3, 1.0))
    p_cogs = p_rev * p_rate

    rev_arr[idx]  = p_rev
    cogs_arr[idx] = p_cogs

    if (i + 1) % 100 == 0 or i == 0:
        print(f"    Day {i+1:>3}/{len(test_dates)}  {pd.Timestamp(tdate).date()}  "
              f"Rev={p_rev:>12,.0f}  Rate={p_rate:.3f}  COGS={p_cogs:>12,.0f}")

In [ ]:
# ─────────────────── 12. EXPORT ───────────────────
submission = test[["Date"]].copy()
submission["Revenue"] = [rev_arr[date_to_idx[pd.Timestamp(d)]]  for d in test["Date"]]
submission["COGS"]    = [cogs_arr[date_to_idx[pd.Timestamp(d)]] for d in test["Date"]]
submission["Revenue"] = submission["Revenue"].round(2)
submission["COGS"]    = submission["COGS"].round(2)
submission["Date"]    = submission["Date"].dt.strftime("%Y-%m-%d")
submission.to_csv(OUTPUT_FILE, index=False)

print(f"\n9. SAVED -> {OUTPUT_FILE}  ({len(submission)} rows)")
print(f"  Revenue: {submission['Revenue'].min():,.0f} -> {submission['Revenue'].max():,.0f}")
print(f"  COGS   : {submission['COGS'].min():,.0f} -> {submission['COGS'].max():,.0f}")
print(submission.head(5).to_string(index=False))